# Демо проектных упражнений модуля 18 — Проверяемое завершение и восстановление после сбоя

**Демо-реализация, не готовое решение полномасштабного проекта.** Сначала наблюдаем исправление аргумента и внешнюю проверку результата. Затем воспроизводим потерю ответа после фиксации операции в SQLite: одинаковый идентификатор предотвращает повторный эффект. Второй опыт действительно читает десять файлов и восстанавливает оставшуюся работу из контрольной точки.
Все входы малы, детерминированы и работают без сети на CPU. Цель — проверить
интерфейсы, метрики, инварианты и структуру отчёта до расходования GPU/API-бюджета.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"воспроизводимый тестовый запуск; seed={SEED}")
DEMO_LANGUAGE = 'ru'


Сначала неверный тип аргумента превращается в наблюдение, и следующая попытка исправляет его. Критерий завершения проверяет и ответ, и результат инструмента. Затем обратите внимание на точку сбоя: SQLite уже зафиксировала запись, а клиент ещё не получил ответ. Один и тот же идентификатор означает повтор прежней операции; новый идентификатор означает новую операцию даже при прежних аргументах.

In [ ]:
import json
import math
from dataclasses import dataclass


@dataclass(frozen=True)
class Tool:
    """Exact scalar contract for a trusted, local teaching handler."""
    arguments: dict
    handler: object


def parse_action(payload):
    def unique(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ValueError("duplicate key")
            result[key] = value
        return result

    def finite(token):
        value = float(token)
        if not math.isfinite(value):
            raise ValueError("non-finite number")
        return value

    if not isinstance(payload, str) or len(payload) > 4096:
        raise ValueError("invalid payload size or type")
    action = json.loads(payload, object_pairs_hook=unique,
                        parse_float=finite, parse_constant=finite)
    if not isinstance(action, dict) or set(action) != {"tool", "args"}:
        raise ValueError("expected exactly tool and args")
    if not isinstance(action["tool"], str) or not isinstance(action["args"], dict):
        raise ValueError("invalid action envelope")
    return action


def react_loop(model, toolbox, query, verify, max_steps=10, max_attempts=6):
    """Bounded read-only teaching loop; completion needs an external verifier."""
    for limit in (max_steps, max_attempts):
        if type(limit) is not int or limit <= 0:
            raise ValueError("budgets must be positive integers")
    events = [{"kind": "user", "text": query}]
    attempts = 0
    for step in range(max_steps):
        response = model(tuple(events))
        if not isinstance(response, str) or len(response) > 4096:
            return {"status": "invalid_response", "events": events}
        if response.startswith("Answer:"):
            answer = response[7:].strip()
            if answer and verify(answer, tuple(events)) is True:
                return {"status": "completed", "answer": answer, "events": events}
            events.append({"kind": "verification_fail"})
            continue
        if not response.startswith("Action:"):
            events.append({"kind": "protocol_error"})
            continue
        if attempts >= max_attempts:
            return {"status": "attempt_budget", "events": events}
        attempts += 1  # Includes malformed and rejected action proposals.
        try:
            action = parse_action(response[7:].strip())
            tool = toolbox.get(action["tool"])
            if tool is None:
                raise ValueError("unknown tool")
            args = action["args"]
            if set(args) != set(tool.arguments):
                raise ValueError("wrong argument names")
            if any(type(args[k]) is not t for k, t in tool.arguments.items()):
                raise TypeError("wrong scalar type")
            result = tool.handler(**args)
            content = json.dumps(result, allow_nan=False)
            if len(content) > 256:
                raise ValueError("observation too long")
            events.append({"kind": "tool_result", "value": result})
        except (ValueError, TypeError, KeyError, ArithmeticError, RecursionError) as error:
            events.append({"kind": "tool_error", "error_type": type(error).__name__})
    return {"status": "max_steps", "events": events}

toolbox = {"double": Tool({"value": float}, lambda value: 2 * value)}
def policy(events):
    values = [e["value"] for e in events if e["kind"] == "tool_result"]
    if values:
        return "Answer: " + str(values[-1])
    if any(e["kind"] == "tool_error" for e in events):
        return 'Action: {"tool":"double","args":{"value":13.96}}'
    return 'Action: {"tool":"double","args":{"value":"13.96"}}'
def verifier(answer, events):
    return answer == "27.92" and any(e.get("value") == 27.92 for e in events)
trace = react_loop(policy, toolbox, "Double a fictional sensor value", verifier)
assert trace["status"] == "completed"
assert [e["kind"] for e in trace["events"]] == ["user","tool_error","tool_result"]
print(json.dumps(trace, indent=2))
assert react_loop(lambda events: "Answer: 0", toolbox, "task", verifier,
                  max_steps=2)["status"] == "max_steps"

# A durable local service commits its effect and operation record atomically.
import sqlite3, tempfile
from contextlib import closing
from pathlib import Path
def service_call(path, operation_id, amount):
    with closing(sqlite3.connect(path)) as db, db:
        db.execute("BEGIN IMMEDIATE")
        row = db.execute("select amount from operations where id=?", (operation_id,)).fetchone()
        if row:
            if row[0] != amount:
                raise ValueError("same operation id with changed arguments")
            return row[0]
        db.execute("insert into operations values (?, ?)", (operation_id, amount))
        db.execute("update counter set value=value+?", (amount,))
    return amount

with tempfile.TemporaryDirectory(dir=Path.cwd()) as folder:
    path = Path(folder)/"service.db"
    with closing(sqlite3.connect(path)) as db, db:
        db.execute("create table operations(id text primary key, amount integer)")
        db.execute("create table counter(value integer)")
        db.execute("insert into counter values (0)")
    service_call(path,"job-1",7)
    # Inject loss AFTER commit and BEFORE a client checkpoint/acknowledgment.
    try:
        raise ConnectionError("committed result was lost")
    except ConnectionError:
        receipt = service_call(path,"job-1",7)
    with closing(sqlite3.connect(path)) as db, db:
        assert db.execute("select value from counter").fetchone()[0] == 7
        assert db.execute("select count(*) from operations").fetchone()[0] == 1
    try:
        service_call(path,"job-1",8)
    except ValueError:
        pass
    else:
        raise AssertionError("changed replay was accepted")
    # A fresh id denotes a fresh operation: a retry with a new id duplicates it.
    service_call(path,"job-2",7)
    with closing(sqlite3.connect(path)) as db, db:
        assert db.execute("select value from counter").fetchone()[0] == 14
print("7 with stable id; 14 after a new id" if DEMO_LANGUAGE=="en" else
      "7 при сохранении идентификатора; 14 после нового идентификатора")

После потери ответа повтор вернул прежний результат, а счётчик остался равным 7. Новый идентификатор увеличил его до 14. Это следствие общей транзакции для журнала и эффекта, а не свойство JSON или локальной отметки клиента. Для стороннего сервиса такие условия нужно проверять отдельно.

Контрольная точка хранит результат вместе с хешем входного файла. Сначала все входы прежние: остаётся выполнить пять работ. Затем один из уже обработанных файлов получает вторую функцию, и мы снова загружаем исходную частичную контрольную точку. Теперь незавершённых или устаревших работ шесть. Сравнивайте восстановленный результат с полным новым анализом тех же файлов.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import ast, hashlib
# Ten actual tiny source files; workers read disjoint inputs and return records.
with tempfile.TemporaryDirectory(dir=Path.cwd()) as folder:
    directory = Path(folder)
    paths = []
    for i in range(10):
        path = directory/f"file_{i}.py"
        path.write_text(f"def value_{i}():\n    return {i}\n", encoding="utf-8")
        paths.append(path)
    def analyze(path):
        payload = path.read_bytes()
        text = payload.decode("utf-8")
        return {"id":path.name,"sha256":hashlib.sha256(payload).hexdigest(),
                "functions":sum(isinstance(n, ast.FunctionDef) for n in ast.walk(ast.parse(text)))}
    sequential = {r["id"]:r for r in map(analyze,paths)}
    checkpoint = directory/"checkpoint.json"
    partial = {r["id"]:r for r in map(analyze,paths[:5])}
    checkpoint.write_text(json.dumps(partial), encoding="utf-8")
    # The coordinator restarts after a simulated failure. Validate input identity.
    restored = json.loads(checkpoint.read_text(encoding="utf-8"))
    pending = [p for p in paths if p.name not in restored or
               restored[p.name]["sha256"] != hashlib.sha256(p.read_bytes()).hexdigest()]
    with ThreadPoolExecutor(max_workers=3) as pool:
        resumed = list(pool.map(analyze,pending))
    for record in resumed:
        # Pending work may replace a stale result for a changed input.
        assert record["id"] not in restored or restored[record["id"]]["sha256"] != record["sha256"]
        restored[record["id"]] = record
    assert len(resumed) == 5 and restored == sequential
    assert sum(r["functions"] for r in restored.values()) == 10
    # Repeat recovery from the same partial checkpoint after changing one saved input.
    paths[0].write_text("def a():\n    return 0\ndef b():\n    return 1\n", encoding="utf-8")
    restored = json.loads(checkpoint.read_text(encoding="utf-8"))
    pending = [p for p in paths if p.name not in restored or
               restored[p.name]["sha256"] != hashlib.sha256(p.read_bytes()).hexdigest()]
    assert len(pending) == 6
    with ThreadPoolExecutor(max_workers=3) as pool:
        updated = list(pool.map(analyze,pending))
    assert len({r["id"] for r in updated}) == len(updated)
    restored.update({r["id"]:r for r in updated})
    fresh = {r["id"]:r for r in map(analyze,paths)}
    assert restored == fresh
    assert sum(r["functions"] for r in restored.values()) == 11
print("5 results reused; 5 jobs resumed; 10 functions verified" if DEMO_LANGUAGE=="en"
      else "5 результатов сохранены; 5 работ выполнены после восстановления; проверены 10 функций")
print("Changed input: 4 results reused; 6 recomputed; 11 functions" if DEMO_LANGUAGE=="en"
      else "Изменённый вход: 4 результата сохранены; 6 пересчитаны; 11 функций")
# These are deterministic Python workers, not autonomous LLM subagents.
# No wall-clock speedup is claimed for this small I/O and AST workload.

При прежних входах получены 10 функций, при изменённом — 11; оба результата совпали с полным анализом. Хеш позволил не спутать наличие записи с её актуальностью. Входы в этом опыте не меняются во время работы исполнителей; согласованный снимок при конкурентном редактировании потребовал бы отдельного механизма. Скорость и качество рассуждения LLM здесь не измеряются.

## Переход к полному проекту

В учебном опыте запись операции и её эффект принадлежат одной транзакции. Для внешнего сервиса нужен его собственный договор повторов; локальный журнал не делает чужую запись атомарной. Полный проект добавляет настоящую модель, MCP, ограничения среды, версии входов, измерение времени и проверку качества. Потоки Python здесь проверяют координацию, а не интеллект субагентов.

### Минимальный отчёт

1. Версии данных, модели, кода и seed.
2. Основная метрика, а также качество, стоимость, задержка и память там, где это применимо.
3. Базовая линия, одинаковый бюджет и хотя бы один отрицательный контроль.
4. Неудачные прогоны и границы обоснованных выводов.